# Experiments 4 — PartNet-Mobility articulation / functional kinematics

**顺序执行 Cell 1–10。** 本 notebook 使用独立的 PartNet-Mobility 原始 URDF 数据集（不是 `E:\datasets\partnet` 的静态 PartNet）。结果只写入 `partnet_results/experiments4_mobility/`，不修改 `experiments2/3` 冻结缓存。

研究分为两层：**A. GT-conditioned kinematics** 对真实 URDF hinge/slider 关节执行同一运动，比较 Anchor、center-based Local、pivot-aware Local（强对照）、frozen Global 的几何表示能力；**B. LLM pivot/axis inference** 只给语义标签和初始 OBB，让共享 DeepSeek 推断关节参数，真实 URDF joint pivot/axis 完全不进 prompt。所有对象一条关节动作测试；多个关节先在对象内平均，避免伪重复。

**注意：** 第一层给所有方法使用真实 GT pivot/axis，是 parameterization capability control，**不是 LLM 生成质量**。Pivot-aware Local 和 Anchor 在该设定下应等价，不能声称 Local 表示无法实现 joint motion。第二层 LLM 的 axis sign 缺少正方向信息，所以同时报告 signed / unsigned alignment；没有真正物理引擎、接触力和动力学。

## Cell 1 — 导入最新 methods.py

In [1]:
import os
import json
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import methods
importlib.reload(methods)
from methods import *

## Cell 2 — 配置数据路径和固定协议（主要只需修改 `mobility_root`）

In [2]:
mobility_root = os.getenv("PARTNET_MOBILITY_ROOT", r"E:\datasets\partnet-mobility")
output_dir = Path("partnet_results") / "experiments4_mobility"
random_seed = 42
max_objects_per_category = 20
max_categories = 8
min_objects_per_category = 3
bootstrap_samples = 3000
run_llm_joint_inference = True
cfg4 = PartNetConfig()
cfg4.model = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")
cfg4.temperature = 0.0
cfg4.allow_api = run_llm_joint_inference
print("Mobility root:", mobility_root)
print("Output:", output_dir.resolve())

Mobility root: E:\datasets\partnet-mobility
Output: D:\MyJupyter\Works\ASPT\partnet_results\experiments4_mobility


## Cell 3 — 发现真实 Mobility 原始资产（不进行重复的 PartNet audit）

In [3]:
assets4 = discover_experiments4_assets(mobility_root)
asset_categories = pd.Series([x["category"] for x in assets4]).value_counts()
print("URDF assets:", len(assets4))
display(asset_categories.to_frame("objects").head(25))

FileNotFoundError: PartNet-Mobility data missing: E:\datasets\partnet-mobility. Set mob_root in Cell 2 to the dataset root containing mobility.urdf.

## Cell 4 — 冻结类别均衡、数据可用的 Mobility 对象清单

一次读取 URDF + 有面引用的 OBJ，只用真实数据是否可解析以及有没有 hinge/slider 关节决定可用性；**不以 Anchor 或任一方法的结果筛选**。以后重新运行将直接读取冻结 manifest。

In [ ]:
manifest4 = build_experiments4_manifest(mobility_root, output_dir, seed=random_seed, max_per_category=max_objects_per_category, max_categories=max_categories, min_per_category=min_objects_per_category)
print("Frozen objects:", len(manifest4["objects"]))
print("Frozen categories:", manifest4["categories"])
display(pd.DataFrame(manifest4["objects"]).groupby("category")[["hinge_count", "slider_count"]].sum())

## Cell 5 — 真实 URDF 前向运动学 + 四种表示更新（离线，无 API）

对同一关节设置 q0→q1，只动目标 child link 及后代。所有方法初始 geometry 完全相同，GT Forward Kinematics 作为参考。真实 pivot 允许在包围盒之外，绝不裁剪。

In [ ]:
kinematics4 = run_experiments4_mobility(manifest4, output_dir)
print("Evaluated objects:", kinematics4["object_count"])
print("Joint events:", kinematics4["event_count"])
print("Method-event records:", len(kinematics4["records"]))

## Cell 6 — 按对象聚合，比较 hinge 与 slider

`center_error` 和 `rotation_error_rad` 相对真实 URDF 运动结果；`joint_point_error` 是真实关节点在预测 child link 运动下的漂移。`global_frozen` 的 hinge pivot 漂移可能为 0，但它根本没有运动，必须结合 pose error 阅读。

In [ ]:
kinematic_summary4 = summarize_experiments4_mobility(kinematics4, output_dir, bootstrap_samples=bootstrap_samples, seed=random_seed)
display(pd.DataFrame(kinematic_summary4["summary"]).round(6))
display(pd.DataFrame(kinematic_summary4["paired_statistics"])[["kind", "baseline", "metric", "n_objects", "mean_improvement", "ci_low", "ci_high", "holm_p", "wins", "ties", "losses"]].round(6))

## Cell 7 — LLM 预测关节轴和枢轴（共享一次，支持断点续跑）

**唯一调用 DeepSeek 的 Cell。** 输入只有类别、link 语义名称、joint kind 和初始化 OBB。**不含**真实轴/枢轴、URDF joint limit 或运动后 GT pose。LLM 错误会保留，不利用 GT 答案自动修复。改 `run_llm_joint_inference=False` 可跳过 API 部分。

In [ ]:
if run_llm_joint_inference:
    predictions4 = run_experiments4_llm_joint_prediction(manifest4, output_dir, cfg4, max_attempts=2)
    prediction_rows = predictions4["events"]
    print("Predicted events:", len(prediction_rows))
    print("Valid JSON + vectors:", sum(x["status"]=="valid" for x in prediction_rows))
else:
    predictions4 = None
    print("LLM inference skipped; offline GT-conditioned result remains available")

## Cell 8 — 从共享 LLM 关节参数执行四种更新，评价真实 pose（无 API）

In [ ]:
if predictions4 is not None:
    llm_scores4 = score_experiments4_llm_joint_prediction(manifest4, predictions4, output_dir)
    print("Valid predicted joint events:", llm_scores4["predicted_joint_events"])
    display(pd.DataFrame(llm_scores4["summary"]).round(6))
else:
    llm_scores4 = None

## Cell 9 — 汇总图和关键方法学限制

In [ ]:
summary_fig4 = pd.DataFrame(kinematic_summary4["summary"])
for kind in sorted(summary_fig4["kind"].unique()):
    subset = summary_fig4[summary_fig4["kind"]==kind]
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.bar(subset["method"], subset["center_error"])
    ax.set_title(f"Mobility {kind}: mean GT link-center error (object level)")
    ax.set_ylabel("Normalized center error ↓")
    ax.tick_params(axis="x", rotation=20)
    fig.tight_layout()
    output_dir.mkdir(parents=True,exist_ok=True)
    fig.savefig(output_dir/f"{kind}_kinematic_center_error.png", dpi=160)
    plt.show()
print("Pivot-aware Local is an explicit positive control; do not interpret equality as Anchor failure.")
print("Out-of-box anchors (descriptive):")
display(summary_fig4[["kind", "method", "both_anchors_inside_fraction"]].drop_duplicates().round(3))

## Cell 10 — 打包结果（上传这一个 zip）

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
required4 = ["benchmark_manifest.json", "kinematics.json", "summary.json"]
if run_llm_joint_inference:
    required4 += ["llm_joint_predictions.json", "llm_pose_results.json"]
archive4 = output_dir / "experiments4_review.zip"
with ZipFile(archive4, "w", compression=ZIP_DEFLATED) as archive:
    for filename in required4:
        path = output_dir / filename
        if not path.is_file():
            raise FileNotFoundError(path)
        archive.write(path, arcname=filename)
    for path in sorted(output_dir.glob("*_kinematic_center_error.png")):
        archive.write(path, arcname=path.name)
print("Upload:", archive4.resolve())
print("MB:", round(archive4.stat().st_size/1024**2, 2))